In [4]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    cross_val_score,
    GroupKFold
)

import optuna
from optuna.samplers import TPESampler

import numpy as np

import os
#os.environ["TQDM_DISABLE"] = "True"
import warnings

In [13]:
def prepare_data(df, categorical_cols, boolean_cols):
    df_new = df.drop(columns = ["timeInterval_ID"])
    X = df_new.drop(columns=['phq8_score', 'participant_name'])
    y = df_new['phq8_score']
    groups = df_new['participant_name']
    cols_to_encode = categorical_cols + boolean_cols
    for col in cols_to_encode:
        X[col] = X[col].cat.codes
        X[col] = X[col].replace(-1, np.nan)
        X[col] = X[col].astype('category')

    return X, y, groups

def get_oh_encoder(all_cat_cols, X):
    categorical_cols_dict = {
        col: sorted(X[col].dropna().unique().tolist())
        for col in all_cat_cols
    }
    categories_list = list(categorical_cols_dict.values())
    oh_encoder = OneHotEncoder(
        categories=categories_list,
        sparse_output=False,
    )
    return oh_encoder

def create_pipeline(X, categorical_cols, boolean_cols, numerical_cols):

    oh_encoder = get_oh_encoder(categorical_cols, X)

    # numeric pipeline
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # categorical pipeline
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", oh_encoder)
    ])

    # boolean pipeline
    boolean_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numerical_cols),
            ("cat", categorical_pipeline, categorical_cols),
            ("bool", boolean_pipeline, boolean_cols),
        ],
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("elasticnet", ElasticNet(random_state=42, max_iter=20000))
    ])

    return pipeline

In [3]:
SEED = 42

In [16]:
def run_optuna_search(X, y, gkf, groups, categorical_cols, all_boolean_cols, numerical_cols, n_trials=40):
    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)

        pipeline = create_pipeline(X, categorical_cols, all_boolean_cols, numerical_cols)

        pipeline.set_params(
            elasticnet__alpha=alpha,
            elasticnet__l1_ratio=l1_ratio
        )

        # Cross-validate with PR-AUC — robust to class imbalance
        neg_mses = cross_val_score(
            pipeline, X, y,
            cv=gkf,                     # stratified cross validiation
            groups = groups,
            scoring="neg_mean_squared_error", # MSE
            n_jobs=-1,                  # parallelize
        )
        return (-1) * neg_mses.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=SEED),
        study_name="depression_elastic_net_tuning",
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

In [5]:
DATA_PATH = "../data_cleaned/final_dataset.parquet"

In [6]:
df = pd.read_parquet(DATA_PATH)

df["any_disorder"] = df["any_disorder"].astype('category')

demographic_cols = [
    'age',
    'gender',
    'marital_status',
    'children',
    'ethnicity_1',
    'drop_school_1',
    'accommodation',
    'mh_smoking',
    'audit_1',
    'depression_medication'
]

categorical_cols = [
    'marital_status',
    'ethnicity_1',
    'accommodation',
    'mh_smoking',
    'audit_1',
]

boolean_cols_demographics = [
    'children',
    'drop_school_1',
    'depression_medication',
    'gender'
]

boolean_cols_sleep = [
    'any_disorder'
]

all_boolean_cols = boolean_cols_demographics + boolean_cols_sleep

numerical_cols = ['age']

group_cols = ['participant_name', 'timeInterval_ID']

outcome_col = ['phq8_score']
    

cols_to_exclude_numerical = demographic_cols + boolean_cols_sleep + group_cols + outcome_col

numerical_cols += [
    col for col in df.columns
    if col not in cols_to_exclude_numerical
]

X, y, groups = prepare_data(df, categorical_cols, boolean_cols_demographics)

In [8]:
gkf = GroupKFold(n_splits=3)

In [17]:
study = run_optuna_search(X, y, gkf, groups, categorical_cols, all_boolean_cols, numerical_cols, n_trials=40)

Best trial: 0. Best value: 33.7627:   8%|▊         | 3/40 [00:08<02:15,  3.65s/it]/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.289e+04, tolerance: 3.384e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.703e+03, tolerance: 3.274e+01
  model = cd_fast.enet_coordinate_descent(
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarnin

In [19]:
best_params = study.best_params
print(f"\nBest MSE (CV): {study.best_value:.4f}")
print(f"\nBest hyperparameters:\n{best_params}\n")


Best MSE (CV): 32.7867

Best hyperparameters:
{'alpha': 0.06687806785055164, 'l1_ratio': 0.644770152763058}

